### Change directory:

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change directory:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

## 1. Update the config.yaml 

Take dataset from ingestion output, transform it using Pegasus tokenizer, and save results in a separate folder.

## 2. Update the params.py

In this case, we dont have any parameters. Whenever we define the model configuration, we usually include parameters, but at this stage, they are not required.

## 3. Update/Create the Entity:

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

## 4. Update the Configuration Manager:

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    # This defines a method inside a class, build configuration for data transformation stage
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation # Reads the data_transformation section from YAML config and config holds all transformation settings.

        create_directories([config.root_dir]) # Ensures the output folder exists. If not, it creates it

        data_transformation_config = DataTransformationConfig( # Builds a structured configuration object. Transfers values from YAML → Python object
            root_dir=config.root_dir, # where transformed data will be saved
            data_path=config.data_path, # input data from ingestion stage
            tokenizer_name = config.tokenizer_name # Hugging Face tokenizer to use
        )

        return data_transformation_config # Sends the configuration to the pipeline, So the next stage (Data Transformation component) can use it


## 5. Update the Components:

In [ ]:
import os
from textsummarizer.logging import logger
from transformers import AutoTokenizer # loads pretrained tokenizer from Hugging Face
from datasets import load_dataset, load_from_disk # loads dataset (from internet or local disk)

In [ ]:
# # This class handles data preprocessing and tokenization
# class DataTransformation:
#     def __init__(self, config: DataTransformationConfig): # This is the constructor. It runs automatically when the class is created. It receives configuration settings (config)
#         self.config = config # Saves the config inside the class. So we can use paths and settings later
#         self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name) # Loads a pretrained tokenizer from Hugging Face. Converts text into numbers (tokens) for the model

#     def convert_examples_to_features(self,example_batch): # This function converts raw text into model-ready format. It processes a batch of data (multiple examples at once)
#         input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True ) # Takes the dialogue (input text). Converts it into numbers (tokens). max_length=1024 → limits long text. truncation=True → cuts extra text if too long. Output = model input format
        
#         with self.tokenizer.as_target_tokenizer(): # Switch to target mode (summary): Tells tokenizer: “now we are processing output text (summary)”. Important for sequence-to-sequence models like Pegasus
#             target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True ) # Encode input text (dialogue):Takes the dialogue (input text), Converts it into numbers (tokens), max_length=1024 → limits long text, truncation=True → cuts extra text if too long. Output = model input format

#         # The dialogue converted into numbers. This is what the model reads. attention_mask, 1 = real words, 0 = padding (ignore this). Helps the model focus only on real text. The summary converted into numbers.  This is the correct answer the model should learn to produce. input_ids → question (dialogue), labels → answer (summary), attention_mask → tells what to ignore    
#         return { # Return final format 
#             'input_ids' : input_encodings['input_ids'], # encoded dialogue (input to model)
#             'attention_mask': input_encodings['attention_mask'], # tells model which words matter (1 = real, 0 = padding)
#             'labels': target_encodings['input_ids'] # correct summary (what model should learn to produce)
#         }
    

#     def convert(self):
#         dataset_samsum = load_from_disk(self.config.data_path) # Loads the dataset from your local folder. This is the output of data ingestion stage
#         dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True) # Apply transformation, Applies your tokenizer function to every example. Converts:dialogue → input_ids, summary → labels, batched=True: means it processes multiple rows at once (faster)
#         dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset")) # Save transformed data: Saves the processed dataset. This data is now ready for model training

In [ ]:
class DataTransformation:

    def __init__(self, config):
        self.config = config

        self.tokenizer = AutoTokenizer.from_pretrained(
            config.tokenizer_name
        )

    def convert_examples_to_features(self, example_batch):

        # Tokenize dialogue
        input_encodings = self.tokenizer(
            example_batch['dialogue'], # Converts input conversation into tokens.
            max_length=1024, 
            truncation=True,
            padding='max_length'
        )

        # Tokenize summary
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'], # Converts target summary into labels.
            max_length=128,
            truncation=True,
            padding='max_length' # Makes all sequences same length.Needed for batching during training.
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):

        print("Loading dataset...")

        dataset_samsum = load_from_disk(
            self.config.data_path
        )

        print("Transforming dataset...")

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        print("Saving transformed dataset...")

        dataset_samsum_pt.save_to_disk(
            self.config.root_dir
        )

        print("Transformation completed successfully.")

## 6. Update the Pipelines:

In [20]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-05-11 14:26:56,792: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 14:26:56,794: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 14:26:56,796: INFO: common: created directory at: artifacts]
[2026-05-11 14:26:56,798: INFO: common: created directory at: artifacts/data_transformation]
[2026-05-11 14:26:57,010: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 14:26:57,028: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-11 14:26:57,115: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 14:26:57,133: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-c

Map: 100%|██████████| 818/818 [00:00<00:00, 3282.13 examples/s]


Saving transformed dataset...


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 91086.11 examples/s] 

Transformation completed successfully.
